## __PHÂN TÍCH HIỆU SUẤT GIAO HÀNG LOGISTIC__

### __1. Mô Tả Dataset__

Bộ dữ liệu ghi lại nhật ký **theo dõi (tracking) các chuyến vận tải đường bộ** của một mạng lưới
logistics tại **Ấn Độ**, chủ yếu phục vụ các nhà máy ô tô/công nghiệp (Daimler, Ford, Ashok Leyland,
Larsen & Toubro…). Mỗi dòng là **một chuyến hàng** với đầy đủ hành trình từ lúc đặt (booking) đến lúc
giao, kèm dữ liệu GPS.

| Thuộc tính | Giá trị |
|---|---|
| Grain (1 dòng = ?) | **1 chuyến hàng**, khóa  Booking ID  |
| Số dòng | **3.585** (còn **3.582** sau khi loại 3 dòng trùng) |
| Số cột | **28** |
| Khoảng thời gian | **15/04/2019 -> 03/12/2020**, nhưng khối lượng dồn vào **06-08/2020** |
| Bối cảnh | Logistics nội địa Ấn Độ (địa danh & tuyến trong nước) |

---

Dữ liệu có thể chia thành **4 nhóm**:

1. **Thông tin vận chuyển** — mã đơn, loại hình, thời điểm booking, các mốc thời gian kế hoạch/thực tế (ETA, trip start/end).
2. **GPS & vận hành** — nhà cung cấp GPS, tọa độ điểm đi/đến/hiện tại, thời điểm ping, quãng đường.
3. **Đối tác & khách hàng** — nhà xe (supplier), khách nhận hàng (customer), tài xế, biển số xe, loại xe.
4. **Vật liệu** — loại hàng hóa được vận chuyển.

Danh sách các cột


| Cột | Mô tả |
|---|---|
| GpsProvider | Nhà cung cấp dịch vụ GPS theo dõi phương tiện. |
| BookingID | Mã định danh duy nhất cho mỗi lần đặt chuyến (booking). |
|  Shipment Type  | Cho biết chuyến là **Market** (đặt lẻ / spot) hay **Regular** (theo hợp đồng). |
|  BookingID_Date  | Ngày và giờ tạo booking. |
|  Vehicle Registration  | Số đăng ký (biển số) duy nhất của xe dùng để vận chuyển. |
|  Origin_Location  | Điểm xuất phát ban đầu của chuyến hàng. |
|  Destination_Location  | Điểm đến cuối cùng nơi hàng được giao. |
|  Origin_loc_latitude  | Vĩ độ của điểm xuất phát. |
|  Origin_loc_longitude  | Kinh độ của điểm xuất phát. |
|  Destination_loc_latitude  | Vĩ độ của điểm đến. |
|  Destination_loc_longitude  | Kinh độ của điểm đến. |
|  Data_Ping_time  | Thời điểm bản ghi GPS (ping) gần nhất từ xe. |
|  Planned_ETA  | Thời gian đến dự kiến tại điểm đến theo kế hoạch chuyến đi. |
|  Current_Location  | Vị trí gần nhất được ghi nhận của xe. |
|  actual_eta  | Thời gian đến thực tế tại điểm đến. |
|  Current_loc_latitude  | Vĩ độ vị trí hiện tại của xe. |
|  Current_loc_longitude  | Kinh độ vị trí hiện tại của xe. |
|  ontime  | Cho biết xe có đến đúng giờ hay không (Yes/No). |
|  trip_start_date  | Ngày và giờ bắt đầu chuyến đi thực tế. |
|  trip_end_date  | Ngày và giờ đến (ước tính). |
|  TRANSPORTATION_DISTANCE_IN_KM  | Tổng quãng đường xe đã đi trong chuyến, tính bằng km. |
|  vehicleType  | Loại xe dùng cho chuyến hàng (ví dụ: 32 FT Truck, Tata Ace, Multi-Axle). |
|  Minimum_kms_to_be_covered_in_a_day  | Quãng đường tối thiểu xe được kỳ vọng đi trong một ngày. |
|  Driver_Name  | Tên tài xế được phân công cho chuyến. |
|  Driver_MobileNo  | Số điện thoại liên hệ của tài xế. |
|  customerNameCode  | Tên khách hàng nhận hàng. |
|  supplierNameCode  | Tên nhà cung cấp / đơn vị gửi hàng (nhà xe). |
|  Material Shipped  | Mô tả loại hàng hóa được vận chuyển trong chuyến. |



### __2. Import thư viện__

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

### __3. Tổng quan dữ liệu__

In [20]:
file_path = '../data/raw/Transportation & Logistics Tracking Dataset.xlsx'
df = pd.read_excel(file_path, sheet_name='Primary Data')
df.head()

,Gps Provider,Booking ID,Shipment Type,Booking Date,Vehicle Registration,Origin Location,Destination Location,Origin Location Latitude,Origin Location Longitude,Destination Location Latitude,...,Trip Start Date,Trip End Date,Transportation Distance (KM),Vehicle Type,Minimum Kms To Be Covered In A Day,Driver Name,Driver Mobile No,Customer Name,Supplier Name,Material Shipped
0,Consent Track,AEIBK2027469,Regular,2020-08-26 12:03:46.000,MH14GD9464,"Shive, Pune, Maharashtra","Pondur, Kanchipuram, Tamil Nadu",18.750621,73.877190,12.930429,...,2020-08-26 16:16:00,2020-08-28 12:15:10,1290.0,32 FT Multi-Axle 14MT - HCV,NaN,VIRAT NILAPALLE,9.960008e+09,Daimler India Commercial Vehicles Pvt Lt,Oms Logistics Pvt Ltd,Regulator - 12v
1,Vamosys,VCV00014153/082021,Regular,2020-08-27 15:21:48.570,TN30BC9320,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000,12.839000,...,2020-08-27 15:21:06,2020-08-27 15:21:54.947000,29.0,NaN,NaN,SENTHIL KUMAR,NaN,Daimler India Commercial Vehicles Pvt Lt,Vj Logistics,Valve Spring
2,Vamosys,VCV00014063/082021,Regular,2020-08-27 14:22:17.833,TN30BB1036,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000,12.839000,...,2020-08-27 14:21:21,2020-08-27 14:22:25.037000,21.0,NaN,NaN,ANBU,NaN,Daimler India Commercial Vehicles Pvt Lt,Vj Logistics,Valve Spring
3,Vamosys,VCV00014741/082021,Regular,2020-08-28 00:32:20.523,TN88D4134,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000,12.839000,...,2020-08-28 00:31:41,2020-08-28 00:32:24.213000,20.0,NaN,NaN,SUDHAKAR,NaN,Daimler India Commercial Vehicles Pvt Lt,Namakkal Sri Anjinaya Transport,Lu Hood Lock / Rh
4,Consent Track,AEIBK2027446,Regular,2020-08-26 09:55:08.000,GJ01DZ8943,"Khorajnanoda, Ahmedabad, Gujarat","Singaperumalkoil, Kanchipuram, Tamil Nadu",22.961777,72.094219,12.786517,...,2020-08-26 15:35:00,2020-08-28 11:21:00,1900.0,32 FT Single-Axle 7MT - HCV,NaN,MAN SINGH,6.396821e+09,Ford India Private Limited,Sterling Translogistics Private Limited,Lu Latch / Pin


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 28 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Gps Provider                        3585 non-null   object        
 1   Booking ID                          3585 non-null   object        
 2   Shipment Type                       3585 non-null   object        
 3   Booking Date                        3585 non-null   datetime64[ns]
 4   Vehicle Registration                3585 non-null   object        
 5   Origin Location                     3585 non-null   object        
 6   Destination Location                3585 non-null   object        
 7   Origin Location Latitude            3585 non-null   float64       
 8   Origin Location Longitude           3585 non-null   float64       
 9   Destination Location Latitude       3585 non-null   float64       
 10  Destination Location Lon

In [18]:
df.shape

(3585, 28)

In [22]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Booking Date,3585,2020-06-21 21:10:33.783650560,2019-04-15 15:15:13,2020-06-12 16:15:00,2020-07-23 13:50:18,2020-08-14 23:34:17.889999872,2020-12-03 13:10:21,NaN
Origin Location Latitude,3585.0,17.870904,9.973636,12.839,16.560192,22.961777,30.000345,5.688552
Origin Location Longitude,3585.0,78.8575,72.056,76.835337,79.632,80.184717,91.843582,4.48121
Destination Location Latitude,3585.0,19.049239,8.172701,12.839,18.660455,23.953847,32.684722,6.390109
Destination Location Longitude,3585.0,78.928819,70.740636,76.835337,78.099421,79.975221,94.961065,4.450874
Data Ping time,3584,2020-06-29 19:31:07.833147392,2019-06-14 15:20:12,2020-06-25 14:52:38.750000128,2020-07-29 17:35:13,2020-08-18 00:05:08,2020-08-28 12:15:10,NaN
Current Location Latitude,3584.0,18.505396,8.70089,12.835129,16.964161,23.283158,32.367928,6.130438
Curren Location Longitude,3584.0,78.947352,69.657698,76.854271,78.20903,80.019062,95.52955,4.291132
Transportation Distance (KM),3437.0,841.10032,0.0,107.0,400.0,1290.0,2898.0,851.889681
Minimum Kms To Be Covered In A Day,940.0,250.531915,250.0,250.0,250.0,250.0,275.0,3.609543


> **Nhận xét:** Dữ liệu có 3.585 dòng × 28 cột. Dữ liệu có các nhóm: thông tin vận chuyển, GPS/tọa độ, đối tác/khách hàng, và vật liệu. Ngày ở file gốc lưu dạng *Excel serial* nên cần ép kiểu ở bước sau.

### __4. Kiểm tra và làm sạch dữ liệu__

In [24]:
df.isnull().sum()

Gps Provider                             0
Booking ID                               0
Shipment Type                            0
Booking Date                             0
Vehicle Registration                     0
Origin Location                          0
Destination Location                     0
Origin Location Latitude                 0
Origin Location Longitude                0
Destination Location Latitude            0
Destination Location Longitude           0
Data Ping time                           1
Planned ETA                              0
Current Location                         0
Actual ETA                              25
Current Location Latitude                1
Curren Location Longitude                1
Ontime                                   0
Trip Start Date                          0
Trip End Date                            0
Transportation Distance (KM)           148
Vehicle Type                           764
Minimum Kms To Be Covered In A Day    2645
Driver Name

In [25]:
df.duplicated().sum()

np.int64(0)

In [ ]:
# Đổi tên cột về dạng snake_case
RENAME = {
    "Gps Provider":"gps_provider","Booking ID":"booking_id","Shipment Type":"shipment_type",
    "Booking Date":"booking_date","Vehicle Registration":"vehicle_reg","Origin Location":"origin",
    "Destination Location":"destination","Origin Location Latitude":"origin_lat",
    "Origin Location Longitude":"origin_lon","Destination Location Latitude":"dest_lat",
    "Destination Location Longitude":"dest_lon","Data Ping time":"data_ping_time",
    "Planned ETA":"planned_eta","Current Location":"current_location","Actual ETA":"actual_eta",
    "Current Location Latitude":"current_lat","Curren Location Longitude":"current_lon",
    "Ontime":"ontime_flag","Trip Start Date":"trip_start","Trip End Date":"trip_end",
    "Transportation Distance (KM)":"distance_km","Vehicle Type":"vehicle_type",
    "Minimum Kms To Be Covered In A Day":"min_kms_per_day","Driver Name":"driver_name",
    "Driver Mobile No":"driver_mobile","Customer Name":"customer_name",
    "Supplier Name":"supplier_name","Material Shipped":"material_shipped",
}

# Ép kiểu ngày và số
DATE_COLS = ["booking_date","planned_eta","actual_eta","trip_start","trip_end","data_ping_time"]
TEXT_COLS = ["gps_provider","shipment_type","vehicle_reg","origin","destination","current_location",
             "ontime_flag","vehicle_type","driver_name","customer_name","supplier_name","material_shipped"]
NUM_COLS  = ["distance_km","origin_lat","origin_lon","dest_lat","dest_lon","current_lat","current_lon"]
NULL_LIKE = re.compile(r"^\s*(NULL|NA|N/A|NAN|NONE|-|\?)\s*$", re.IGNORECASE)

# Chuẩn hóa chuỗi "NULL"/"NA"/"-" -> NaN thật, trim khoảng trắng
for c in TEXT_COLS:
    df[c] = df[c].astype("string").str.strip()
    df[c] = df[c].mask(df[c].str.match(NULL_LIKE, na=False))
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["ontime_flag"] = df["ontime_flag"].str.title()

# distance_km = 0 là vô lý -> coi như thiếu
df.loc[df["distance_km"] == 0, "distance_km"] = np.nan

# Bo 2 cột không dùng
df = df.drop(columns=["min_kms_per_day", "driver_mobile"], errors="ignore")

# Gán "Unknown" cho biến phân loại để nhóm
for c in ["vehicle_type", "driver_name"]:
    df[c] = df[c].fillna("Unknown")

miss = df.isna().sum(); miss = miss[miss > 0].sort_values(ascending=False)
print("Dữ liệu còn thiếu sau khi làm sạch:"); print(miss.to_string() if len(miss) else "(Không còn)")

Dữ liệu còn thiếu sau khi làm sạch:
distance_km         166
actual_eta           25
current_location     12
data_ping_time        1
gps_provider          1
current_lat           1
current_lon           1


>Nhận xét chất lượng dữ liệu:

Bộ dữ liệu khá sạch về mặt cấu trúc — các cột khóa (`booking_id`, `origin`, `destination`, `ontime_flag`, `customer_name`, `supplier_name`) đều đầy đủ 100%. Vấn đề thiếu dữ liệu tập trung ở vài cột và được xử lý theo 3 hướng:

- Bỏ cả cột (thiếu quá nhiều / không giá trị phân tích): min_kms_per_day thiếu ~74% và driver_mobile là PII → loại bỏ.
- Giữ NaN, loại khỏi phép tính liên quan (biến đo lường — tuyệt đối không điền để tránh bịa số): actual_eta thiếu 25 dòng → 25 chuyến không tính được delay; distance_km thiếu 148 dòng và có giá trị 0 bất hợp lý → quy về NaN, loại khi tính tốc độ.
Gán "Unknown" (biến phân loại để nhóm): vehicle_type (764) và driver_name (317) → giữ lại các chuyến này trong biểu đồ thay vì để chúng biến mất.
- Về trùng lặp: nếu chỉ xét trùng trên mọi cột thì df.duplicated().sum() = 0. Nhưng xét theo khóa nghiệp vụ booking_id thì có 2 mã trùng / 3 dòng thừa — thực chất là các chuyến chở nhiều loại hàng cho nhiều khách (consolidated), không phải bản sao lỗi. Vì grain đã chốt là 1 dòng = 1 chuyến, ta gộp về 1 dòng/booking (ảnh hưởng < 0,1% dữ liệu).

Kết luận: dữ liệu đủ tin cậy để phân tích. 

### __5. Feature Engineering - Tạo các cột để phân tích__

Dữ liệu thô chỉ ghi lại sự kiện (các mốc thời gian, tọa độ, tên đối tác), chưa trả lời được câu hỏi phân tích. Ví dụ: file gốc có `planned_eta` và `actual_eta`, nhưng **không có cột nào nói "chuyến này trễ bao lâu"** — muốn phân tích delay thì buộc phải tự tính. Bước 3 biến dữ liệu thô thành các cột **đo lường được** và **nhóm được**, chia làm 3 loại:

- **Metric (đo lường)** — không có sẵn, phải tính: `delay_hours` (= `actual_eta − planned_eta`), `trip_duration_hours`, `avg_speed_kmph`.
- **Flag / phân loại** — để đếm tỷ lệ & gán nhãn: `is_delayed` (0/1, dùng tính *delay rate*), `delivery_status`, `delay_level` (Slight/Medium/Severe), `is_delay_outlier`.
- **Dimension (chiều cắt lát)** — để nhóm dữ liệu: `route` (ghép origin + destination), `booking_month`, `booking

In [44]:
WEEKDAY = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

df['delay_hours'] = ((df['actual_eta'] - df['planned_eta']).dt.total_seconds() / 3600).round(2)
df['is_delayed'] = np.where(df['delay_hours'] > 0, 1, 0)
df.loc[df['delay_hours'].isna(), 'is_delayed'] = np.nan 
df['delivery_status'] = np.select(
    [df['delay_hours'].isna(), df['delay_hours'] > 0],
    ['Unknown', 'Delayed'], default='On Time')

def lvl(h):
    if pd.isna(h): return 'Unknown'
    if h <= 0: return 'On Time'
    if h <= 24: return 'Slight Delay'
    if h <= 72: return 'Medium Delay'
    return 'Severe Delay'

# Mức độ delay
df['delay_level'] = df['delay_hours'].apply(lvl)
lo, hi = df['delay_hours'].quantile([0.01, 0.99])
df['is_delayed_outlier'] = ((df['delay_hours'] < lo )| (df['delay_hours'] > hi)).astype('Int64')

# Tuyến đường
df['route'] = df['origin'].fillna('Unknown') + '->' + df['destination'].fillna('Unknown')

# Thời gian chuyến đi
df['trip_duration_hours'] = ((df['trip_end'] - df['trip_start']).dt.total_seconds() / 3600).round(2)
ok = (df['trip_duration_hours'] > 0) & (df['distance_km'] > 0)

# Tốc độ trung bình
df['avg_speed_kmph'] = np.where(ok, (df['distance_km'] / df['trip_duration_hours']).round(2), np.nan)

# Thời gian booking theo tháng và tuần
df["booking_month"]   = df["booking_date"].dt.to_period("M").dt.to_timestamp()
df["booking_weekday"] = pd.Categorical(df["booking_date"].dt.day_name(), categories=WEEKDAY, ordered=True)

print("Số cột sau feature engineering:", df.shape[1])
df[["booking_id","route","distance_km","delay_hours","delivery_status","delay_level"]].head()



Số cột sau feature engineering: 36


,booking_id,route,distance_km,delay_hours,delivery_status,delay_level
0,AEIBK2027469,"Shive, Pune, Maharashtra->Pondur, Kanchipuram,...",1290.0,-51.26,On Time,On Time
1,VCV00014153/082021,"Daimler India Commercial Vehicles,Kanchipuram,...",29.0,-79.13,On Time,On Time
2,VCV00014063/082021,"Daimler India Commercial Vehicles,Kanchipuram,...",21.0,-78.82,On Time,On Time
3,VCV00014741/082021,"Daimler India Commercial Vehicles,Kanchipuram,...",20.0,-89.01,On Time,On Time
4,AEIBK2027446,"Khorajnanoda, Ahmedabad, Gujarat->Singaperumal...",1900.0,-30.76,On Time,On Time


### __6. Kiểm chứng dữ liệu__

In [45]:
ct = pd.crosstab(df['ontime_flag'], df['delivery_status'])
ct

delivery_status,Delayed,On Time,Unknown
ontime_flag,,,
No,2187,1,16
Yes,0,1372,9


In [46]:
match = ((df.ontime_flag=="Yes")&(df.delivery_status=="On Time")).sum() + \
        ((df.ontime_flag=="No")&(df.delivery_status=="Delayed")).sum()
valid = df.delay_hours.notna().sum()
print(f"\nKhớp {match}/{valid} dòng có ETA hợp lệ ({match/valid*100:.1f}%)")


Khớp 3559/3560 dòng có ETA hợp lệ (100.0%)


> Nhận xét: khớp ~99,9%. → Có thể dùng delay_hours làm chuẩn.